# Línea base: clasificador de reclamos de clientes
**Proyecto:** clasificar mensajes en `pagos`, `envios`, `cuenta` u `otro`.
**Este notebook:** mide un modelo preentrenado **sin ajustar** (zero-shot). Ese resultado es la línea base que el modelo ajustado con LoRA debe superar.

Datos: ficticios (sintéticos). No contienen información de personas reales.

## 1. Cargar los datos
Sube `mensajes_sinteticos.csv` cuando Colab te lo pida.

In [1]:
import pandas as pd
try:
    from google.colab import files
    files.upload()   # selecciona mensajes_sinteticos.csv
except ImportError:
    pass
df = pd.read_csv("mensajes_sinteticos.csv")
print(df.shape)
df.categoria.value_counts()

Saving mensajes_sinteticos.csv to mensajes_sinteticos.csv
(298, 6)


,count
categoria,
cuenta,75
otro,75
envios,75
pagos,73


## 2. Separar entrenamiento y prueba
Cada mensaje base tiene varias variantes (columna `grupo`). Se divide **por grupo** para que una variante en prueba nunca tenga a su gemela en entrenamiento. Si se dividiera al azar, el acierto saldría inflado.

In [2]:
from sklearn.model_selection import GroupShuffleSplit
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(gss.split(df, df.categoria, groups=df.grupo))
train, test = df.iloc[idx_train], df.iloc[idx_test]
print(len(train), "entrenamiento |", len(test), "prueba")
print(test.categoria.value_counts().to_dict())
train.to_csv("train.csv", index=False); test.to_csv("test.csv", index=False)

239 entrenamiento | 59 prueba
{'envios': 18, 'pagos': 17, 'otro': 15, 'cuenta': 9}


## 3. Modelo de referencia (sin ajustar)
Un modelo de inferencia en lenguaje natural que clasifica texto sin entrenamiento previo en la tarea. Se le dan descripciones de cada categoría en español.

In [ ]:
from transformers import pipeline
clf = pipeline("zero-shot-classification", model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

ETIQUETAS = {
    "pagos":  "problemas con pagos, cobros, reembolsos o facturas",
    "envios": "problemas con el envío, la entrega o el estado de un pedido",
    "cuenta": "problemas con la cuenta, la contraseña o el acceso",
    "otro":   "consultas generales, sugerencias o felicitaciones",
}
nombres, descripciones = list(ETIQUETAS), list(ETIQUETAS.values())

def predecir(texto):
    r = clf(texto, candidate_labels=descripciones,
            hypothesis_template="Este mensaje trata sobre: {}.")
    return nombres[descripciones.index(r["labels"][0])]

test = test.copy()
test["prediccion"] = [predecir(t) for t in test.mensaje]
test.head()

config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  558MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.26k [00:00<?, ?B/s]

... (2 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 16.3MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

## 4. Resultados

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print(classification_report(test.categoria, test.prediccion, labels=nombres, zero_division=0))
cm = confusion_matrix(test.categoria, test.prediccion, labels=nombres)
ConfusionMatrixDisplay(cm, display_labels=nombres).plot(cmap="Blues")
plt.title("Línea base (sin ajustar)"); plt.show()

## 5. El error que más importa
Un reclamo **urgente de pagos** que se va a `otro` queda sin atender.

In [ ]:
pagos_urg = test[(test.categoria == "pagos") & (test.urgente == "si")]
perdidos = pagos_urg[pagos_urg.prediccion == "otro"]
print(f"Pagos urgentes en prueba: {len(pagos_urg)}")
print(f"Enviados a 'otro':        {len(perdidos)}")
pagos_otro = test[(test.categoria == "pagos") & (test.prediccion == "otro")]
print(f"Cualquier pago enviado a 'otro': {len(pagos_otro)}")
test[test.categoria != test.prediccion][["mensaje", "categoria", "prediccion"]].head(15)

## 6. Apuntes para la clase
- Acierto global de la línea base: _(escríbelo)_
- Categoría con peor resultado: _(escríbelo)_
- Pagos urgentes que se van a `otro`: _(escríbelo)_
- ¿Qué patrón ves en los errores? ¿Falla la etiqueta, el contexto o el modelo?